# SatQuery AI — ConvNeXt scene classifier (single file, Colab or Kaggle)

**One file, run top to bottom.** Fine-tunes a ConvNeXt-tiny scene/land-cover
classifier on the SatQuery BigEarthNet subset and prints validation
accuracy. Do not run this on a local machine — it needs a GPU and the
dataset, neither of which are in the repo.

Works on **either Google Colab or Kaggle** — the config cell below
auto-detects which one it's running on.

## Option A — Google Colab (use this if you already have the dataset in
## Google Drive, e.g. `/content/drive/MyDrive/SatQueryAI/dataset`)

1. Open this notebook in Colab (**File → Upload notebook**).
2. **Runtime → Change runtime type → T4 GPU**.
3. **Run All.** The config cell mounts your Drive and will prompt you to
   authorize access the first time — that's normal, just approve it.
4. If your Drive folder isn't at `/content/drive/MyDrive/SatQueryAI/dataset`,
   edit `DATASET_DIR` in the config cell to match.

## Option B — Kaggle

1. Zip the dataset folder containing `training.csv` and
   `BENv2_lithuania_summer.lmdb/` (with `data.mdb` + `lock.mdb`) and upload
   it as a new Kaggle Dataset (**Datasets → New Dataset**, Kaggle
   auto-extracts the zip).
2. Upload this notebook (**File → Upload Notebook**), then
   **Add Input → Datasets** and attach the dataset from step 1.
3. **Settings → Accelerator → GPU T4**.
4. Edit `DATASET_DIR` in the config cell to match your attached dataset's
   input path (shown in the Kaggle sidebar, usually
   `/kaggle/input/<your-dataset-slug>`).
5. **Run All.**

## After it finishes (either platform)

Download `bentxt_convnext.pt` (Colab: left-hand Files panel, under
`/content/output`; Kaggle: the notebook's Output tab) and place it at
`models/bentxt_convnext.pt` in your local repo checkout.
`backend/app/llm/local_classifier.py` loads it from there automatically —
no other changes needed.

The inspect cell below prints `type`/`category` value counts from
`training.csv` before filtering. If nothing in that printout obviously
means "scene/land-cover classification" for this CSV, edit
`CATEGORY_KEYWORDS` / `TYPE_KEYWORDS` in the config cell and re-run from
there.


## Config — auto-detects Colab vs Kaggle, edit `DATASET_DIR` if needed


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET_DIR = "/content/drive/MyDrive/SatQueryAI/dataset"  # <-- edit if your Drive folder differs
    OUTPUT_DIR = "/content/output"
else:
    DATASET_DIR = "/kaggle/input/satqueryai-dataset"  # <-- edit to your attached Kaggle dataset's path
    OUTPUT_DIR = "/kaggle/working"

CSV_PATH = f"{DATASET_DIR}/training.csv"
LMDB_PATH = f"{DATASET_DIR}/BENv2_lithuania_summer.lmdb"
CHECKPOINT_PATH = f"{OUTPUT_DIR}/bentxt_convnext.pt"

# Which type/category values (case-insensitive substring match) count as
# "scene/land-cover classification" rows, as opposed to VQA/grounding/
# bounding-box rows. Check the inspect cell's output and adjust if needed.
CATEGORY_KEYWORDS = ["land cover", "land-cover", "landcover", "scene", "classif", "dominant", "predominant"]
TYPE_KEYWORDS: list[str] = []
LABEL_COLUMN = "output"

MIN_LABEL_COUNT = 20   # drop classes with fewer than this many examples
MAX_ROWS = 4000        # cap dataset size for a fast fine-tune
VAL_FRAC = 0.15
SEED = 42

MODEL_NAME = "convnext_tiny"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LR = 3e-4
NUM_WORKERS = 2
UNFREEZE_LAST_BLOCK = False

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Platform: {'Colab' if IN_COLAB else 'Kaggle/other'}")
print(f"DATASET_DIR = {DATASET_DIR}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")


## Install dependencies not preinstalled by default (Colab or Kaggle)


In [ ]:
!pip install -q timm lmdb safetensors


## LMDB helpers

Sentinel-2 true-color bands in the reBEN/BigEarthNet-v2 safetensors sample are `B04`/`B03`/`B02` (red/green/blue), Sentinel-1 fallback is `VV`. If your LMDB uses different keys, the inspect-samples cell below prints one sample's keys so you can fix `S2_RED`/`S2_GREEN`/`S2_BLUE`/`S1_PRIMARY` here.

In [ ]:
import logging
import numpy as np

logger = logging.getLogger(__name__)

S2_RED, S2_GREEN, S2_BLUE = "B04", "B03", "B02"
S1_PRIMARY = "VV"


def open_lmdb_env(lmdb_path: str):
    import lmdb
    return lmdb.open(lmdb_path, readonly=True, lock=False, readahead=False, max_readers=126)


def load_patch_sample(env, patch_id: str):
    from safetensors.numpy import load as safetensor_load
    with env.begin(write=False) as txn:
        raw = txn.get(str(patch_id).encode())
    if raw is None:
        return None
    return safetensor_load(raw)


def _percentile_stretch(band: np.ndarray, lo: float = 2.0, hi: float = 98.0) -> np.ndarray:
    finite = band[np.isfinite(band)]
    if finite.size == 0:
        return np.zeros_like(band, dtype=np.uint8)
    p_lo, p_hi = np.percentile(finite, [lo, hi])
    if p_hi <= p_lo:
        return np.zeros_like(band, dtype=np.uint8)
    scaled = np.clip((band - p_lo) / (p_hi - p_lo), 0, 1)
    return (scaled * 255).astype(np.uint8)


def sample_to_rgb_uint8(sample: dict) -> np.ndarray:
    keys = set(sample.keys())
    if {S2_RED, S2_GREEN, S2_BLUE} <= keys:
        r = _percentile_stretch(np.asarray(sample[S2_RED]).astype(np.float64))
        g = _percentile_stretch(np.asarray(sample[S2_GREEN]).astype(np.float64))
        b = _percentile_stretch(np.asarray(sample[S2_BLUE]).astype(np.float64))
        return np.stack([r, g, b], axis=-1)
    if S1_PRIMARY in keys:
        gray = _percentile_stretch(np.asarray(sample[S1_PRIMARY]).astype(np.float64))
        return np.stack([gray, gray, gray], axis=-1)
    first_key = sorted(keys)[0]
    logger.warning("Unrecognized band keys %s; falling back to '%s'.", keys, first_key)
    gray = _percentile_stretch(np.asarray(sample[first_key]).astype(np.float64))
    return np.stack([gray, gray, gray], axis=-1)


## Step 1 — load + inspect `training.csv`

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows, {df['patch_id'].nunique()} unique patch_ids.")
print("\n--- df[\'type\'].value_counts() ---")
print(df["type"].value_counts(dropna=False))
print("\n--- df[\'category\'].value_counts() ---")
print(df["category"].value_counts(dropna=False))
print(
    "\nIf nothing above obviously means scene/land-cover classification, "
    "edit CATEGORY_KEYWORDS / TYPE_KEYWORDS in the config cell and re-run from there."
)


## Step 2 — filter to classification rows, dedupe, drop rare labels, subset

In [ ]:
import numpy as np

cat = df["category"].astype(str).str.lower()
typ = df["type"].astype(str).str.lower()
cat_mask = cat.apply(lambda v: any(k in v for k in CATEGORY_KEYWORDS)) if CATEGORY_KEYWORDS else pd.Series(False, index=df.index)
type_mask = typ.apply(lambda v: any(k in v for k in TYPE_KEYWORDS)) if TYPE_KEYWORDS else pd.Series(False, index=df.index)
filtered = df[cat_mask | type_mask].copy()
print(f"After type/category filter: {len(filtered)} rows / {filtered['patch_id'].nunique()} patches")
assert not filtered.empty, "No rows matched — adjust CATEGORY_KEYWORDS/TYPE_KEYWORDS above based on Step 1's printout."

filtered[LABEL_COLUMN] = filtered[LABEL_COLUMN].astype(str).str.strip()

# Keep exactly one QA row per patch_id (its `output` becomes the class label).
deduped = (
    filtered.sample(frac=1.0, random_state=SEED)
    .drop_duplicates(subset="patch_id", keep="first")
    .reset_index(drop=True)
)
print(f"After one-row-per-patch dedupe: {len(deduped)} rows")

counts = deduped[LABEL_COLUMN].value_counts()
keep_labels = counts[counts >= MIN_LABEL_COUNT].index
cleaned = deduped[deduped[LABEL_COLUMN].isin(keep_labels)].copy()
print(f"After dropping labels with < {MIN_LABEL_COUNT} examples: {len(cleaned)} rows, {cleaned[LABEL_COLUMN].nunique()} classes")
assert not cleaned.empty, "Nothing left after dropping rare labels — lower MIN_LABEL_COUNT above."

if len(cleaned) > MAX_ROWS:
    frac = MAX_ROWS / len(cleaned)
    subset = (
        cleaned.groupby(LABEL_COLUMN, group_keys=False)
        .apply(lambda g: g.sample(frac=frac, random_state=SEED) if len(g) > 1 else g)
        .reset_index(drop=True)
    )
else:
    subset = cleaned
print(f"After subsetting to <= {MAX_ROWS} rows: {len(subset)} rows")

LABELS = sorted(subset[LABEL_COLUMN].unique())
print(f"{len(LABELS)} classes: {LABELS}")


## Step 3 — leak-free patch-level train/val split + LMDB join check

In [ ]:
patch_ids = subset["patch_id"].unique()
rng = np.random.default_rng(SEED)
rng.shuffle(patch_ids)
n_val = max(1, int(len(patch_ids) * VAL_FRAC))
val_ids = set(patch_ids[:n_val])

train_df = subset[~subset["patch_id"].isin(val_ids)].copy()
val_df = subset[subset["patch_id"].isin(val_ids)].copy()

overlap = set(train_df["patch_id"]) & set(val_df["patch_id"])
assert not overlap, f"Leaky split: {len(overlap)} patch_ids in both train and val."
print(f"Train: {len(train_df)} rows / {train_df['patch_id'].nunique()} patches")
print(f"Val:   {len(val_df)} rows / {val_df['patch_id'].nunique()} patches")
print("Zero patch_id overlap between train/val confirmed.")

_env = open_lmdb_env(LMDB_PATH)
_sample_ids = subset["patch_id"].drop_duplicates().sample(n=min(20, subset["patch_id"].nunique()), random_state=0)
_missing = 0
_first_keys = None
for _pid in _sample_ids:
    _s = load_patch_sample(_env, _pid)
    if _s is None:
        _missing += 1
    elif _first_keys is None:
        _first_keys = sorted(_s.keys())
print(f"LMDB join check: {_missing}/{len(_sample_ids)} sampled patch_ids missing from LMDB.")
if _first_keys is not None:
    print(f"Sample tensor keys (verify against S2_RED/S2_GREEN/S2_BLUE/S1_PRIMARY above): {_first_keys}")


## Step 4 — Dataset + model

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class BigEarthPatchDataset(Dataset):
    def __init__(self, dataframe, lmdb_path, labels, label_column, img_size):
        self.df = dataframe.reset_index(drop=True)
        self.lmdb_path = lmdb_path
        self._env = None
        self.label_column = label_column
        self.label2idx = {label: i for i, label in enumerate(labels)}
        self.img_size = img_size

    @property
    def env(self):
        # Opened lazily so each DataLoader worker (forked after __init__) gets its own handle.
        if self._env is None:
            self._env = open_lmdb_env(self.lmdb_path)
        return self._env

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample = load_patch_sample(self.env, row["patch_id"])
        if sample is None:
            raise KeyError(f"patch_id {row['patch_id']} missing from LMDB")
        rgb = sample_to_rgb_uint8(sample)
        tfm = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
        image = tfm(rgb)
        label = self.label2idx[str(row[self.label_column])]
        return image, label


def build_model(model_name, num_classes, unfreeze_last_block):
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    for p in model.parameters():
        p.requires_grad = False
    head = getattr(model, "head", None) or getattr(model, "fc", None)
    if head is not None:
        for p in head.parameters():
            p.requires_grad = True
    if unfreeze_last_block:
        stages = getattr(model, "stages", None)
        if stages is not None and len(stages) > 0:
            for p in stages[-1].parameters():
                p.requires_grad = True
    return model


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. Settings > Accelerator > GPU T4.")

train_ds = BigEarthPatchDataset(train_df, LMDB_PATH, LABELS, LABEL_COLUMN, IMG_SIZE)
val_ds = BigEarthPatchDataset(val_df, LMDB_PATH, LABELS, LABEL_COLUMN, IMG_SIZE)
print(f"Train examples: {len(train_ds)}, Val examples: {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model = build_model(MODEL_NAME, len(LABELS), UNFREEZE_LAST_BLOCK).to(device)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)


## Step 5 — fine-tune

In [ ]:
import time
import torch.nn.functional as F


def run_epoch(model, loader, optimizer, device, train: bool):
    model.train(mode=train)
    total_loss, total_correct, total_n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_n += images.size(0)
    return total_loss / max(total_n, 1), total_correct / max(total_n, 1)


for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, optimizer, device, train=False)
    print(
        f"epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f} train_acc={train_acc:.3f}  "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}  ({time.time() - t0:.1f}s)"
    )


## Step 6 — save checkpoint

In [ ]:
import os

os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_name": MODEL_NAME,
        "labels": LABELS,
        "img_size": IMG_SIZE,
        "mean": IMAGENET_MEAN,
        "std": IMAGENET_STD,
    },
    CHECKPOINT_PATH,
)
print(f"Saved checkpoint to {CHECKPOINT_PATH} ({os.path.getsize(CHECKPOINT_PATH)} bytes)")


## Step 7 — validation accuracy + per-class breakdown

Should be well above the chance-level baseline printed alongside it.

In [ ]:
from collections import Counter

model.eval()
correct, total = 0, 0
per_class_correct: Counter = Counter()
per_class_total: Counter = Counter()
with torch.no_grad():
    for images, targets in val_loader:
        images, targets = images.to(device), targets.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
        for p, t in zip(preds.tolist(), targets.tolist()):
            per_class_total[t] += 1
            if p == t:
                per_class_correct[t] += 1

acc = correct / max(total, 1)
chance = 1.0 / max(len(LABELS), 1)
print(f"Validation accuracy: {acc:.4f} over {total} examples ({len(LABELS)} classes, chance={chance:.4f})")
print("\nPer-class accuracy:")
for idx, label in enumerate(LABELS):
    n = per_class_total.get(idx, 0)
    c = per_class_correct.get(idx, 0)
    if n:
        print(f"  {label:30s} {c}/{n} = {c/n:.3f}")


## Done — download the checkpoint

Grab `bentxt_convnext.pt` (Colab: left-hand Files panel, under
`/content/output`; Kaggle: the notebook's Output tab) and place it at
`models/bentxt_convnext.pt` in your local repo.
`backend/app/llm/local_classifier.py` picks it up automatically.
